In [1]:
import xarray as xr, numpy as np, pandas as pd, os, glob
import dask.dataframe as dd
from dask.distributed import Client

In [2]:
client = Client(n_workers=14, threads_per_worker=1, memory_limit='4GB')
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33507 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/33507/status,
Dashboard: /proxy/33507/status,Workers: 14
Total threads: 14,Total memory: 52.15 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45507,Workers: 0
Dashboard: /proxy/33507/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43695,Total threads: 1
Dashboard: /proxy/43139/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:42489,


In [3]:
ehf_fpath = '/scratch/ng72/ms5578/ehf_netcdf'
nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
write_path = '/scratch/ng72/ms5578/'
netcdf_files = glob.glob(os.path.join(ehf_fpath, "*.nc")) 

In [4]:
gen_df = pd.read_csv(f"{nmap_path}/gen_info.csv")

In [5]:
template_ds = xr.open_dataset(netcdf_files[0], engine='netcdf4')
lat_grid = template_ds['lat'].values
lon_grid = template_ds['lon'].values
gen_lats = gen_df['lat'].values
gen_lons = gen_df['lon'].values

# Check for and filter out NaN values in generator coordinates
valid_mask = ~(np.isnan(gen_lats) | np.isnan(gen_lons))
if not valid_mask.all():
    print(f"Warning: Found {(~valid_mask).sum()} generators with NaN coordinates, skipping them")
    gen_lats = gen_lats[valid_mask]
    gen_lons = gen_lons[valid_mask]
    gen_df_filtered = gen_df[valid_mask].reset_index(drop=True)
else:
    gen_df_filtered = gen_df

# Normalize longitudes to [-180, 180) to avoid dateline mismatches
def _normalize_lon(arr):
    a = np.asarray(arr)
    return ((a + 180) % 360) - 180

lon_grid = _normalize_lon(lon_grid)
gen_lons = _normalize_lon(gen_lons)

# Create mesh of grid points (lat, lon) and flatten
lonv, latv = np.meshgrid(lon_grid, lat_grid)  # shapes (nlat, nlon)
lat_flat = latv.ravel()
lon_flat = lonv.ravel()

# Filter out any NaN values from grid coordinates
grid_valid_mask = ~(np.isnan(lat_flat) | np.isnan(lon_flat))
if not grid_valid_mask.all():
    print(f"Warning: Found {(~grid_valid_mask).sum()} grid points with NaN coordinates")
    lat_flat = lat_flat[grid_valid_mask]
    lon_flat = lon_flat[grid_valid_mask]
    # Store mapping from filtered indices back to original grid indices
    original_indices = np.arange(len(grid_valid_mask))[grid_valid_mask]
else:
    original_indices = np.arange(len(lat_flat))

# Convert to radians for haversine metric
grid_rads = np.vstack([np.radians(lat_flat), np.radians(lon_flat)]).T
gen_rads = np.vstack([np.radians(gen_lats), np.radians(gen_lons)]).T

try:
    # Prefer BallTree (true haversine on the sphere)
    from sklearn.neighbors import BallTree
    tree = BallTree(grid_rads, metric='haversine')
    dist, idx_flat = tree.query(gen_rads, k=1)
    idx_flat = idx_flat.ravel()
    method_used = "BallTree"
except Exception:
    # Fallback: use scipy.spatial.cKDTree on 3D unit vectors (approx spherical nearest)
    from scipy.spatial import cKDTree
    def _sph2cart(lat_rad, lon_rad):
        clat = np.cos(lat_rad)
        return np.column_stack([clat * np.cos(lon_rad), clat * np.sin(lon_rad), np.sin(lat_rad)])
    grid_xyz = _sph2cart(np.radians(lat_flat), np.radians(lon_flat))
    gen_xyz = _sph2cart(np.radians(gen_lats), np.radians(gen_lons))
    tree = cKDTree(grid_xyz)
    dist, idx_flat = tree.query(gen_xyz, k=1)
    method_used = "cKDTree"

print(f"Used {method_used} for nearest neighbor search")

# Map filtered indices back to original grid if needed
if not grid_valid_mask.all():
    idx_flat = original_indices[idx_flat]

# Convert flattened indices back to (lat_index, lon_index) in the 2D grid
lat_indices, lon_indices = np.unravel_index(idx_flat, (lat_grid.size, lon_grid.size))

# If we filtered generators, create full-size arrays with NaN for invalid generators
if not valid_mask.all():
    full_lat_indices = np.full(len(gen_df), np.nan)
    full_lon_indices = np.full(len(gen_df), np.nan)
    full_lat_indices[valid_mask] = lat_indices
    full_lon_indices[valid_mask] = lon_indices
    lat_indices = full_lat_indices.astype(float)
    lon_indices = full_lon_indices.astype(float)
    print(f"Final result: {len(gen_df)} total generators, {valid_mask.sum()} with valid coordinates")

Used BallTree for nearest neighbor search
Final result: 500 total generators, 497 with valid coordinates


In [6]:
# Filter out generators with NaN indices (from invalid coordinates)
valid_indices_mask = ~(np.isnan(lat_indices) | np.isnan(lon_indices))
valid_lat_indices = lat_indices[valid_indices_mask].astype(int)
valid_lon_indices = lon_indices[valid_indices_mask].astype(int)
valid_gen_df = gen_df[valid_indices_mask].reset_index(drop=True)

print(f"Processing {len(valid_gen_df)} generators with valid grid assignments")

isel_dict = {
    'lat': xr.DataArray(valid_lat_indices, dims='points'),
    'lon': xr.DataArray(valid_lon_indices, dims='points'),
}

all_ddfs = []

for file in netcdf_files:
    ds = xr.open_dataset(file, engine='netcdf4', chunks='auto')
    sub_ds = ds.isel(lat=isel_dict['lat'], lon=isel_dict['lon'])
    sub_ds = sub_ds.assign_coords(DUID=('points', valid_gen_df['DUID'].values))
    df = sub_ds[['tas','EHF_val', 'HW_EHF_avg', 'HW_EHF_peak', 'EHF_flag','tas_3d_avg','tas_3d_peak']].to_dask_dataframe().reset_index()
    all_ddfs.append(df)


full_ddf = dd.concat(all_ddfs)

Processing 497 generators with valid grid assignments


In [7]:
def clean_and_process(df):
    df['time'] = dd.to_datetime(df['time'])
    # First localize naive timestamps to UTC, then convert to Australia/Brisbane
    df['time'] = df['time'].dt.tz_localize('UTC').dt.tz_convert('Australia/Brisbane')
    df = df.replace(1.000000e+20, np.nan)
    df = df.drop(columns=['height', 'crs','points','index'], errors='ignore')
    df = df.sort_values(by=['DUID', 'time'])
    return df

In [8]:
def number_heatwave_days(df):
    df = df.sort_values(by='time')
    is_hw = df['EHF_flag'] == 1
    event_id = (is_hw != is_hw.shift()).cumsum()
    df['event_group'] = np.where(is_hw, event_id, pd.NA)
    df['HW_event_day'] = df.groupby('event_group').cumcount() + 1
    df.loc[df['event_group'].isna(), 'HW_event_day'] = pd.NA
    return df

In [9]:
df = clean_and_process(full_ddf).compute()
df = df.groupby('DUID', group_keys=False).apply(number_heatwave_days)

/jobfs/156016945.gadi-pbs/ipykernel_476454/2015718923.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('DUID', group_keys=False).apply(number_heatwave_days)


In [10]:
df.to_csv(f"{write_path}/gen_hw_status.csv", index=False)
gen_df.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/nmap.csv', index=False)

In [11]:
# import pandas
# write_path = '/scratch/ng72/ms5578/time_series'
# pandas.read_csv(f"{write_path}/gen_hw_status.csv")